# OntologyRAG-Q — best configuration + Chain-of-Thought + one-shot, run with Gemma

Runs the best-performing setup from Table 4 of Al-Azani et al. (EMNLP 2025):

**Ayat-Ontology chunking · k = 6 · Similarity · temperature = 0.0 · multilingual-e5-small**

with Gemma as the generator, and adds two things on top of it: **chain-of-thought (CoT)
reasoning** and **one-shot prompting** (one fully worked example shown before the real
question).

### This run answers **50 questions** (`N_EVAL_SAMPLES = 50`)

50 Direct questions, one person, no merging. About one GPU hour of answering once the
search index is built, so it finishes inside a single Kaggle session.

### What the chain-of-thought adds

Retrieval is left exactly as the paper's best row — same chunking, same embedder, same
k = 6, same cosine similarity, same greedy decoding. The only thing that changes is the
*generation* stage:

1. The k retrieved passages are numbered `[المقطع 1] … [المقطع 6]` so the model can
   refer to them while reasoning.
2. The model is asked to write its reasoning under `التفكير:` — identify what the
   question asks, scan each numbered passage, pick the ones that actually contain the
   answer, pull out the supporting sentences with the Ayah quoted verbatim, and check
   every claim is grounded (otherwise answer `لا أعرف`).
3. It then writes the answer under `الإجابة النهائية:`.
4. Only the text after `الإجابة النهائية:` is scored. The reasoning is stripped out and
   saved separately, so BLEU / CHRF / BERTScore stay comparable with the paper's numbers.

### What the one-shot example adds

The CoT instructions describe the two sections in words. One-shot prompting *shows* them:
before the real question, the model is given a complete solved case — its own Context and
Question as a user turn, and the reply that was wanted as an assistant turn. Instruction-
tuned models copy the shape of a demonstrated reply far more reliably than the shape of a
described one, which matters here because a reply that never reaches
`الإجابة النهائية:` cannot be scored at all.

The example is **built from real data, never written by hand**:

- a real Direct question, taken from a held-out pool;
- the passages the retriever actually returns for it, cut down to the 3 most relevant;
- its own reference answer, as the final answer;
- four reasoning lines filled in from the question's own record (surah, verse, question
  type), the passage number whose wording really contains the answer, and a sentence
  quoted from that passage.

Writing a demonstration by hand would mean inventing a Tafsir sentence — the one thing
the paper's prompt forbids the model from doing. A model imitates what it is shown, so a
fabricated example teaches fabrication.

**The example is never scored.** `SHOT_POOL_SIZE = 25` questions are removed from the
evaluation pool *in every mode*, one-shot or not. That gives two guarantees:

- the question whose reference answer appears in the prompt is never one of the 50 being
  graded — otherwise the model would be handed the answer it is measured on;
- a zero-shot run and a one-shot run at the same seed answer the identical 50 questions,
  so the difference between their scores is the prompt and nothing else.

Both are re-checked after the run by the sanity-check cell, which fails loudly rather
than quietly reporting a leaked number.

### The four settings you can compare

`USE_COT` and `USE_ONE_SHOT` are independent, so the notebook produces a clean 2×2:

| mode | prompt |
|------|--------|
| `base` | the paper's prompt, no example |
| `base_1shot` | the paper's prompt + one worked example |
| `cot` | reason, then answer |
| `cot_1shot` | reason, then answer + one worked example ← **this notebook** |

Results are written to one file per mode, so nothing is overwritten. Flip the two flags
and Run All to fill in another row; the last-but-one cell prints all the rows that exist
at the same `n`, with each one's difference from the paper baseline.

---

### Setup on Kaggle (do this first)

1. Go to https://huggingface.co/google/gemma-3-4b-it and click **Acknowledge license**.
2. Go to https://huggingface.co/settings/tokens and create a token, type **Read**. Copy it.
3. In this notebook: **Add-ons → Secrets → Add a new secret**. Label it `HF_TOKEN`, paste the token, tick the checkbox.
4. **Settings → Accelerator → GPU**.
5. Use the **same Hugging Face account** for steps 1 and 2.

### How to run

Leave `QUICK_TEST = True` and press **Run All**. This takes about 30 minutes (most of it
building the search index once) and answers 5 questions, just to prove everything works.

Then set `QUICK_TEST = False` and Run All again. The index is cached, so it goes straight
to answering the 50 evaluation questions.

If it stops early, just Run All again — it continues from where it stopped.

### If the session ends before the run does

Answers are saved to `results/preds_..._cot_1shot_n50_seed42.json` after **every single
question**, and the run skips anything already in that file. So stopping is safe — Run All
again and it carries on. But the file has to still exist next time:

**Way 1 — Persistence (easiest).** In the notebook editor's right sidebar, set
**Persistence → Files only**. That keeps `/kaggle/working` between sessions.

**Way 2 — an attached Dataset (always works).** Download
`results/preds_gemma-3-4b-it_all_cot_1shot_n50_seed42.json` before the session ends,
upload it as a **Dataset**, **Add Input** it next time, and **Run All**. The notebook
searches `/kaggle/input` as well as its own folder and prints what it picked up:
`[recovered] 30 answers from the attached dataset: /kaggle/input/...`

The same trick works for the search index: put `chunks_all.json` and `index_all.faiss`
in the dataset too and skip the ~40-minute build every session.

### If your power cuts out

The run is on Kaggle's servers, not your machine, so a cut at your end deletes nothing.
Saving is written to survive a cut *during* a save, which is the dangerous moment: every
file is written to a temp file, flushed to disk, then renamed into place — renames are
atomic, so the real file is never half-written. The predictions file also keeps a `.bak`
copy, and the loader falls back to it if the newer file is damaged. The search index is
written the same way, and a damaged index is detected and rebuilt instead of crashing.

Worst case you lose the single question that was being answered when the power went.

In [ ]:
# ================= CONFIG =================
QUICK_TEST = False                      # True = 5 questions. Set False for the real run.

# ---- chain-of-thought ----
USE_COT = True                          # True = reason step by step, then answer.
                                        # False = the paper's original single-shot prompt.

# The CoT answer is the text after 'الإجابة النهائية:'. Without a length instruction the
# model tends to dump its whole reasoning into that section, which inflates the answer
# and costs BLEU against the short reference answers. This asks for one short paragraph.
# Turn it off if you want the reasoning step measured with no length pressure at all.
CONCISE_FINAL_ANSWER = True

# ---- one-shot prompting ----
# One fully worked example is placed before the real question, as a completed earlier
# turn of the conversation: its own Context and Question, then the reply that was wanted
# for it. Instructions alone leave the model guessing how long the answer should be and
# how the two CoT sections should look; one example shows it.
USE_ONE_SHOT = True
N_SHOT = 1                              # 1 = one-shot. 2+ makes it few-shot and adds
                                        # roughly one context block per extra example.

# The example is built from real data -- a real question, the passages the retriever
# actually returns for it, and its own reference answer. Nothing in it is invented, so
# the model is never shown a Tafsir sentence that does not exist.
#
# The questions it may be drawn from are held out of the evaluation set, in EVERY mode,
# so (a) the example is never one of the questions being scored, and (b) a zero-shot run
# and a one-shot run answer exactly the same questions, which is what makes the two
# numbers comparable.
SHOT_POOL_SIZE = 25                     # questions reserved for the example
SHOT_MAX_PASSAGES = 3                   # passages shown in the example (of TOP_K found)
SHOT_CHARS_PER_CHUNK = 400              # each truncated to this, to keep the prompt small
SHOT_MIN_REF_WORDS = 3                  # a two-word answer demonstrates nothing
SHOT_MAX_REF_WORDS = 40                 # a 200-word one teaches the model to ramble

SHOW_TRACES = 2                         # how many full reasoning traces to print at the end

MODEL_ID = "google/gemma-3-4b-it"       # or "google/gemma-3-1b-it"
QUANT_BITS = 4                          # 8 = better quality, may not fit on a T4

N_EVAL_SAMPLES = 50                     # this notebook answers 50 questions
SEED = 42

# ---- optional: split a run across team members ----
# N_SHARDS = 1 means this notebook answers all N_EVAL_SAMPLES questions itself. That is
# the setup here: 50 questions, one person, no merging.
#
# To split a bigger set between two people instead, both of you set the SAME
# N_EVAL_SAMPLES (e.g. 100) and N_SHARDS = 2, and change only SHARD -- 1 takes the
# first half, 2 takes the second. The set is drawn once and then sliced, so the halves
# cannot overlap, and the merge cell further down combines them.
#
# Note if your teammate runs their own separate 50 with N_SHARDS = 1: with the same
# SEED they get the SAME 50 questions as you, not different ones. For different
# questions they need a different SEED, or use the SHARD split above.
SHARD = 1
N_SHARDS = 1


def shard_bounds(n, shard, n_shards):
    """Contiguous, non-overlapping split of n items; any remainder goes to the
    earliest shards. Used by both the config print and the answering cell, so the
    two can never disagree about who owns which questions."""
    base, extra = divmod(n, n_shards)
    start = (shard - 1) * base + min(shard - 1, extra)
    return start, start + base + (1 if shard - 1 < extra else 0)

# Which books to search.
#   "all"    -- all 15 Tafsir books (55,471 chunks). Harder search.
#   "source" -- only the 2 books the answers were written from (~7,500 chunks).
#
# The paper's Limitations says "we only performed the analysis using one
# source", which is ambiguous. If they searched one book, their search was
# far easier than searching all 15, which would explain part of their high
# scores. Running both settings tells you how much the corpus size matters.
CORPUS = "all"

# The paper's best row -- do not change these. CoT and the one-shot example change the
# generator only; retrieval is byte for byte the paper's.
TOP_K = 6
EMBED_MODEL_ID = "intfloat/multilingual-e5-small"

MAX_CHARS_PER_CHUNK = 2000
MAX_PROMPT_TOKENS = 8192
# Reasoning is generated before the answer, so CoT needs a bigger budget -- with 512 the
# model can run out of tokens mid-reasoning and never reach 'الإجابة النهائية:'.
# Raising the cap is nearly free: generation time depends on the tokens actually
# produced, and the model stops on its own when it is done. The cap only costs
# anything on the answers that would otherwise have been cut off mid-sentence.
MAX_NEW_TOKENS = 1024 if USE_COT else 512
COT_RETRY_NEW_TOKENS = 1536             # one retry, only when the first run hit the cap
BERTSCORE_MODEL = "bert-base-multilingual-cased"
RESULTS_DIR = "results"

# The mode is part of every filename, so the four prompt settings -- baseline, CoT,
# baseline + example, CoT + example -- never overwrite each other's results.
SHOT_TAG = ("" if not USE_ONE_SHOT else
            "_1shot" if N_SHOT == 1 else f"_{N_SHOT}shot")
MODE = ("cot" if USE_COT else "base") + SHOT_TAG

if not 1 <= SHARD <= N_SHARDS:
    raise ValueError(f"SHARD must be between 1 and N_SHARDS ({N_SHARDS}), got {SHARD}.")
if USE_ONE_SHOT and not 1 <= N_SHOT <= SHOT_POOL_SIZE:
    raise ValueError(f"N_SHOT must be between 1 and SHOT_POOL_SIZE "
                     f"({SHOT_POOL_SIZE}), got {N_SHOT}.")

if QUICK_TEST:
    N_EVAL_SAMPLES = 5
    N_SHARDS, SHARD = 1, 1          # a smoke test always runs the whole 5
    print("QUICK_TEST on: 5 questions only, no sharding.")

SHARD_TAG = f"_shard{SHARD}of{N_SHARDS}" if N_SHARDS > 1 else ""
print(f"{MODEL_ID} | {QUANT_BITS}-bit | k={TOP_K} | corpus={CORPUS} "
      f"| n={N_EVAL_SAMPLES} | mode={MODE}{SHARD_TAG and ' | ' + SHARD_TAG[1:]}")
print("prompt: " + ("chain-of-thought" if USE_COT else "paper baseline")
      + (f" + {N_SHOT} worked example(s)" if USE_ONE_SHOT else " + no examples (zero-shot)"))

if N_SHARDS > 1:
    _s, _e = shard_bounds(N_EVAL_SAMPLES, SHARD, N_SHARDS)
    _mine = _e - _s
    print(f"\nYou are shard {SHARD} of {N_SHARDS}: questions {_s + 1}-{_e} "
          f"of the shared {N_EVAL_SAMPLES} ({_mine} to answer).")
    print(f"Your teammate runs the identical notebook with SHARD = "
          f"{2 if SHARD == 1 else 1}.")
else:
    _mine = N_EVAL_SAMPLES or 2350

# 50 questions at roughly 45-75 s each fits inside one Kaggle session with room to
# spare, once the index is built. Anything much bigger does not, so warn about it.
print(f"\n{_mine} questions at roughly 45-75 s each is about "
      f"{_mine * 60 / 3600:.1f} GPU hours of answering, plus a one-off ~40 min to build "
      "the search index.")
if _mine > 200:
    print("Kaggle stops a session at 12 h, so this needs several sittings. The run "
          "resumes from")
    print(f"  {RESULTS_DIR}/preds_*_{MODE}_n{N_EVAL_SAMPLES}_seed{SEED}{SHARD_TAG}.json")
    print("but only if that file survives. Turn on Settings > Persistence "
          "(Variables and Files),")
    print("or download the file before each session ends -- otherwise every "
          "session starts from zero.")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.makedirs(RESULTS_DIR, exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception as e:
    print(f"Could not read HF_TOKEN: {type(e).__name__}")
    print("Fix: Add-ons > Secrets > new secret labelled exactly HF_TOKEN, checkbox ticked.")

!pip install -q -U transformers accelerate bitsandbytes faiss-cpu sentence-transformers openpyxl sacrebleu bert-score tqdm

In [ ]:
# ========== CHECK EVERYTHING BEFORE THE SLOW STEPS ==========
import torch
ok = True

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"[OK]   GPU: {p.name}, {p.total_memory/1e9:.1f} GB")
else:
    ok = False
    print("[FAIL] No GPU. Settings > Accelerator > GPU, then restart the session.")

token = os.environ.get("HF_TOKEN")
if not token:
    ok = False
    print("[FAIL] HF_TOKEN missing -- see the cell above.")
else:
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    try:
        print(f"[OK]   Token belongs to: {api.whoami().get('name')}")
    except Exception:
        ok = False
        print("[FAIL] Token rejected -- create a fresh token of type 'Read'.")
    try:
        api.model_info(MODEL_ID)
        print(f"[OK]   Access to {MODEL_ID} confirmed.")
    except Exception:
        ok = False
        print(f"[FAIL] No access to {MODEL_ID}.")
        print(f"       Open https://huggingface.co/{MODEL_ID}, click 'Acknowledge license',")
        print("       logged in as the user printed above.")

print("\nReady to run." if ok else "\nFix the [FAIL] items before continuing.")

In [ ]:
# ========== LOAD THE MODEL ==========
# Gemma-3-4B is a multimodal checkpoint and needs Gemma3ForConditionalGeneration.
# Gemma-3-1B is text-only and needs a plain causal-LM class. Try each in turn.
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM

bnb = (BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                          bnb_4bit_quant_type="nf4") if QUANT_BITS == 4 else
       BitsAndBytesConfig(load_in_8bit=True) if QUANT_BITS == 8 else None)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

classes = []
for name in ["Gemma3ForConditionalGeneration", "AutoModelForImageTextToText"]:
    try:
        classes.append(getattr(__import__("transformers", fromlist=[name]), name))
    except (ImportError, AttributeError):
        pass
classes.append(AutoModelForCausalLM)

model, errs = None, []
for cls in classes:
    try:
        model = cls.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
        print(f"Loaded with {cls.__name__} ({QUANT_BITS}-bit).")
        break
    except Exception as e:
        errs.append(f"{cls.__name__}: {type(e).__name__}: {str(e)[:200]}")

if model is None:
    for e in errs:
        print(" -", e)
    raise RuntimeError("Could not load the model -- see errors above.")
model.eval()

# Some chat templates accept a separate system turn, some do not. Detect it.
try:
    tokenizer.apply_chat_template([{"role": "system", "content": "x"},
                                   {"role": "user", "content": "y"}],
                                  tokenize=False, add_generation_prompt=True)
    SUPPORTS_SYSTEM_ROLE = True
except Exception:
    SUPPORTS_SYSTEM_ROLE = False
print(f"System role supported: {SUPPORTS_SYSTEM_ROLE}")

In [ ]:
# ========== DOWNLOAD THE DATA ==========
import glob
import json, requests
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


def find_file(fname):
    """Find a cached file in this session, or in any attached Kaggle dataset.

    Kaggle's Persistence setting is the easy way to keep files between sessions, but it
    is not always available. Searching /kaggle/input as well means a run can be carried
    across sessions by hand instead: download the file, upload it as a Dataset, attach
    it with 'Add Input', and the next session picks it up from there.
    """
    if os.path.exists(fname):
        return fname
    hits = sorted(glob.glob(f"/kaggle/input/**/{os.path.basename(fname)}",
                            recursive=True))
    return hits[0] if hits else None

BASE = "https://raw.githubusercontent.com/sazani/OntologyRAG-Q/main"
ALL_BOOKS = [
    "aashoor_v2.xlsx", "alaloosi_v2.xlsx", "almawirdee_v2.xlsx", "almuyassar_v2.xlsx",
    "alrazi_v2.xlsx", "altasheel_v2.xlsx", "aysaraAltafasir_v2.xlsx", "fathAlqadeer_v2.xlsx",
    "fathaAlbayan_v2.xlsx", "katheer_v2.xlsx", "mukhtasar_v2.xlsx", "qurtubi_v2.xlsx",
    "saadi_v2.xlsx", "tabari_v2.xlsx", "zadAlmaseer_v2.xlsx",
]
# The two books the reference answers were written from.
SOURCE_BOOKS = ["aysaraAltafasir_v2.xlsx", "almuyassar_v2.xlsx"]

BOOK_FILES = ALL_BOOKS if CORPUS == "all" else SOURCE_BOOKS
CHUNKS_PATH = f"chunks_{CORPUS}.json"
INDEX_PATH = f"index_{CORPUS}.faiss"

os.makedirs("books", exist_ok=True)
for fname in BOOK_FILES:
    p = os.path.join("books", fname)
    if not os.path.exists(p):
        r = requests.get(f"{BASE}/Resources/Tafaser/Tafaser_DS2/{fname}")
        r.raise_for_status()
        open(p, "wb").write(r.content)

if not os.path.exists("OntologyQA_v1.json"):
    r = requests.get(f"{BASE}/Resources/OntologyQA_v1.json")
    r.raise_for_status()
    open("OntologyQA_v1.json", "wb").write(r.content)

with open("OntologyQA_v1.json", encoding="utf-8") as f:
    qa_data = json.load(f)

chunks = index = embed_model = None
chunks_src, index_src = find_file(CHUNKS_PATH), find_file(INDEX_PATH)
if chunks_src and index_src:
    # A cache written when the machine lost power can be truncated. Rebuilding costs
    # ~40 minutes; crashing the notebook with a raw exception costs more, because it is
    # not obvious that the fix is to delete two files.
    try:
        with open(chunks_src, encoding="utf-8") as f:
            chunks = json.load(f)
        index = faiss.read_index(index_src)
        if index.ntotal != len(chunks):
            raise ValueError(f"{index.ntotal} vectors for {len(chunks)} chunks")
        embed_model = SentenceTransformer(EMBED_MODEL_ID)
        where = ("this session" if chunks_src == CHUNKS_PATH
                 else f"attached dataset ({os.path.dirname(index_src)})")
        print(f"Cached: {len(chunks)} chunks, {index.ntotal} vectors — from {where}.")
    except Exception as e:
        chunks = index = None
        print(f"[warn] The cached index is damaged ({type(e).__name__}: {e}).")
        print("[warn] Rebuilding it below. This is what a power cut during the save "
              "looks like.")

if chunks is None:
    print(f"{len(BOOK_FILES)} books ready, {len(qa_data)} QA pairs. Building index below.")

In [ ]:
# ========== AYAT-ONTOLOGY CHUNKING ==========
# One chunk per verse (or verse range), with the ontology fields -- surah name,
# surah number, verse range -- written into the chunk text, as the paper describes.
#
# NOTE: aysaraAltafasir_v2.xlsx uses different column names from the other 14
# books. Handling only the majority schema silently drops all 1,290 of its rows,
# and that book is the source of ~90% of the Direct questions' answers. Both
# schemas are handled here. With CORPUS="all" this gives 55,471 chunks, which
# is exactly the total reported in the paper's Table 6.
import pandas as pd

STANDARD = {"surah": "SURA_num", "start": "Verse_Number_start",
            "end": "Verse_Number_end", "verse": "passages", "tafsir": "Tafsir"}
AYSARA = {"surah": "SURA_num", "aya_range": "AYA_num",
          "verse": "Ayah", "tafsir": "Tafsir"}
SCHEMAS = {"aysaraAltafasir_v2.xlsx": AYSARA}


def _parse_aya_range(v):
    """'217-218' -> (217, 218); '14-15-16' -> (14, 16); '7' -> (7, 7)."""
    nums = [int(p) for p in str(v).split("-") if p.strip().isdigit()]
    return (min(nums), max(nums)) if nums else (None, None)


if chunks is None:
    surah_names = {}
    for d in qa_data:
        sid, nm = d.get("Sura_ID"), d.get("SURA_name")
        if sid is not None and nm:
            try:
                surah_names[int(sid)] = nm
            except (ValueError, TypeError):
                pass

    chunks, per_book = [], {}
    for fname in BOOK_FILES:
        src = fname.replace("_v2.xlsx", "")
        sc = SCHEMAS.get(fname, STANDARD)
        df = pd.read_excel(os.path.join("books", fname))
        made = 0

        for _, row in df.iterrows():
            su = row.get(sc["surah"])
            if pd.isna(su):
                continue
            if "aya_range" in sc:
                st, en = _parse_aya_range(row.get(sc["aya_range"]))
            else:
                st, en = row.get(sc["start"]), row.get(sc["end"])
            if st is None or en is None or pd.isna(st) or pd.isna(en):
                continue

            tafsir = str(row.get(sc["tafsir"], "")).strip()
            if not tafsir or tafsir == "nan":
                continue
            verse = str(row.get(sc["verse"], "")).strip()

            su, st, en = int(su), int(st), int(en)
            sname = surah_names.get(su, f"سورة {su}")
            chunks.append({
                "surah_number": su, "surah_name": sname,
                "start_ayah": st, "end_ayah": en, "source": src,
                "chunk_text": (f"سورة: {sname} (رقم {su})\n"
                               f"الآية رقم {st} إلى {en}: {verse}\n"
                               f"التفسير ({src}): {tafsir}"),
            })
            made += 1

        per_book[src] = made
        print(f"{src:20} {len(df):5} rows -> {made:5} chunks")

    empty = [b for b, m in per_book.items() if m == 0]
    if empty:
        raise RuntimeError(f"No chunks produced from {empty} -- column schema mismatch.")

    # Written to a temp file and renamed, so a power cut cannot leave a half-written
    # cache that looks valid to the next run.
    with open(CHUNKS_PATH + ".tmp", "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False)
        f.flush()
        os.fsync(f.fileno())
    os.replace(CHUNKS_PATH + ".tmp", CHUNKS_PATH)

    print(f"\nTOTAL: {len(chunks)} chunks", end="  ")
    if CORPUS == "all":
        print("-- matches the paper's Table 6 total (55,471)." if len(chunks) == 55471
              else "-- expected 55,471 per the paper's Table 6; check before trusting results.")
    else:
        print("(source books only)")
else:
    print(f"Skipped -- {len(chunks)} chunks cached.")

In [ ]:
# ========== BUILD THE SEARCH INDEX (the slow step) ==========
# normalize_embeddings + IndexFlatIP together == cosine similarity,
# which is the "Similarity" search in the paper's best row.
if index is None:
    embed_model = SentenceTransformer(EMBED_MODEL_ID)
    # e5 models need "passage: " on stored text and "query: " on searches.
    texts = ["passage: " + c["chunk_text"] for c in chunks]
    print(f"Embedding {len(texts)} chunks. This is the slow part -- do not interrupt.")
    emb = np.array(embed_model.encode(texts, batch_size=64, show_progress_bar=True,
                                      normalize_embeddings=True), dtype="float32")
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    # Same temp-then-rename as the chunks file: 40 minutes of embedding should not be
    # lost to a power cut during the final write.
    faiss.write_index(index, INDEX_PATH + ".tmp")
    os.replace(INDEX_PATH + ".tmp", INDEX_PATH)
    print("Index built:", index.ntotal, "chunks. Cached for next time.")
else:
    print(f"Skipped -- {index.ntotal} vectors cached.")

In [ ]:
# ====== PROMPTS: PAPER BASELINE + CHAIN OF THOUGHT + ONE-SHOT EXAMPLE ======
import re
import unicodedata

# The baseline prompt is copied word for word from the paper, §4.
SYSTEM_PROMPT = """You are an expert in interpreting the Quran, specifically
designed to answer users' questions. Provide answers solely based on the
context provided below. Do not draw upon any external or prior knowledge or
information. If the answer is not found within the given context, respond
with 'I don't know.' Ensure that the Ayah (verses) are quoted verbatim as
they appear in the Quran. All answers should be provided in Arabic."""

# The CoT prompt keeps every constraint of the baseline word for word -- grounding,
# 'I don't know', verbatim Ayat, Arabic -- and only adds the reasoning procedure and a
# fixed two-section output format. The section marker is what lets the reasoning be
# stripped before scoring, so it has to be stated explicitly and never translated.
_COT_STEPS = """
Think step by step before you answer. Your response must contain exactly two
sections, with these two Arabic headings written exactly as shown, each on its own
line, and nothing at all before the first heading. Do not shorten 'الإجابة النهائية'
and do not replace it with any other word:

التفكير:
١. حدد بدقة ما يسأل عنه السؤال: السورة، ورقم الآية، والمفهوم المطلوب.
٢. افحص كل مقطع من مقاطع السياق المرقّمة، واذكر أرقام المقاطع التي تتضمن الإجابة
   فعليًا (استخدم أرقام المقاطع المعطاة فقط، من ١ إلى ٦)، وتجاهل ما لا يتعلق بالسؤال.
٣. اذكر بإيجاز ما يدعم الإجابة في تلك المقاطع.
٤. تحقق من أن كل جزء من إجابتك مذكور صراحة في السياق. إن لم يكن كذلك، فالإجابة
   هي 'لا أعرف'.

قيود هذا القسم: أربع نقاط، سطر واحد لكل نقطة. لا تنسخ نصوصًا طويلة من السياق هنا؛
النقل الحرفي للآيات يكون في الإجابة النهائية، لا في التفكير. القسم الأهم هو الإجابة
النهائية، فلا تستهلك المساحة في التفكير.

الإجابة النهائية:
{answer_rule}"""

_ANSWER_RULE_CONCISE = """اكتب هنا الإجابة النهائية بالعربية فقط، في فقرة واحدة موجزة
مكتفية بذاتها، بأسلوب كتب التفسير. لا تذكر أرقام المقاطع، ولا تشر إلى السياق أو إلى
خطوات التفكير، ولا تكرر السؤال."""

_ANSWER_RULE_PLAIN = """اكتب هنا الإجابة النهائية بالعربية فقط، مكتفية بذاتها. لا تذكر
أرقام المقاطع، ولا تشر إلى السياق أو إلى خطوات التفكير."""

SYSTEM_PROMPT_COT = SYSTEM_PROMPT + "\n" + _COT_STEPS.format(
    answer_rule=_ANSWER_RULE_CONCISE if CONCISE_FINAL_ANSWER else _ANSWER_RULE_PLAIN)

# The worked example is shown as a completed earlier turn, not pasted into the question,
# so the model sees exactly the shape of reply it is being asked to produce. This note
# says what that turn is for. Without it the model sometimes answers the example's
# question instead of the real one, or carries the example's Ayah into an unrelated
# answer -- which would be an ungrounded answer, the one thing the paper's prompt forbids.
_ONE_SHOT_NOTE_TEMPLATE = """

The conversation starts with {count}: a Context and a Question, followed by
the reply that was expected for them. Imitate that reply's FORMAT, its section headings
and its length. Never reuse its CONTENT: the example's passages, verses and answer
belong to its own question, and must not appear in your answer unless the new Context
itself contains them. Answer only the last question."""

_ONE_SHOT_NOTE = _ONE_SHOT_NOTE_TEMPLATE.format(
    count="one solved example" if N_SHOT == 1 else f"{N_SHOT} solved examples")


def system_prompt_for(cot, one_shot):
    """The system prompt actually sent, for a given mode. One place, so the example and
    the real question can never end up under different instructions."""
    return (SYSTEM_PROMPT_COT if cot else SYSTEM_PROMPT) + (_ONE_SHOT_NOTE if one_shot else "")


ACTIVE_SYSTEM_PROMPT = system_prompt_for(USE_COT, USE_ONE_SHOT)

# Matches 'الإجابة النهائية:' and the spellings the model actually drifts to --
# hamza-less 'الاجابة', 'الجواب النهائي', an English fallback, markdown bold or a
# heading around it, and a missing colon.
_STRICT_FINAL = re.compile(
    r"(?:^|\n)\s*(?:#{1,6}\s*)?(?:\*{1,3}|_{1,2})?\s*"
    r"(?:الإجابة\s*النهائية|الاجابة\s*النهائية|الإجابه\s*النهائيه|"
    r"الجواب\s*النهائي|Final\s*Answer)"
    r"\s*(?:\*{1,3}|_{1,2})?\s*[:：]?\s*")

# Gemma often shortens the heading to just 'الإجابة:' or 'الخلاصة:'. Those are only
# tried when the full heading is absent, and a colon is required, so an ordinary
# sentence containing the word 'الجواب' cannot be mistaken for a heading.
_LOOSE_FINAL = re.compile(
    r"(?:^|\n)\s*(?:#{1,6}\s*)?(?:\*{1,3}|_{1,2})?\s*"
    r"(?:الإجابة|الاجابة|الجواب|الخلاصة|Answer)"
    r"\s*(?:\*{1,3}|_{1,2})?\s*[:：]\s*")

# Lines that look like reasoning steps: '1.', '٢)', '- ', '* '.
_STEP_LINE = re.compile(r"^\s*(?:[*\-–]|[0-9٠-٩]{1,2}[\.\)])\s+", re.M)

_THINK_MARKER = re.compile(
    r"(?:^|\n)\s*(?:#{1,6}\s*)?(?:\*{1,3}|_{1,2})?\s*"
    r"(?:التفكير|التحليل|خطوات\s*التفكير|Reasoning|Thinking)"
    r"\s*(?:\*{1,3}|_{1,2})?\s*[:：]?\s*")


def _tidy(text):
    """Drop leftover markdown and stray leading punctuation from a parsed section."""
    text = re.sub(r"\*{2,3}|_{2,}|^#{1,6}\s*", "", text.strip())
    return re.sub(r"^[\s:：\-–—•\.]+", "", text).strip()


def _last_paragraph(text):
    paras = [p for p in re.split(r"\n\s*\n", text.strip()) if p.strip()]
    return _tidy(paras[-1]) if paras else ""


def split_cot(raw, truncated=False):
    """Split a CoT generation into (reasoning, final answer, parsed_ok, reason).

    Only the final answer is scored, so everything here is about not putting the
    wrong text in it. `reason` records which route was taken, so a long run can be
    audited afterwards instead of just counting failures.
    """
    # 1. The heading we asked for, then the shortened forms Gemma drifts to.
    for pat, reason in ((_STRICT_FINAL, "ok"), (_LOOSE_FINAL, "short_heading")):
        matches = list(pat.finditer(raw))
        if not matches:
            continue
        m = matches[-1]                      # last one: the model sometimes restates it
        reasoning, answer = raw[:m.start()], _tidy(raw[m.end():])
        tm = _THINK_MARKER.search(reasoning)
        if tm:
            reasoning = reasoning[tm.end():]
        if answer:
            return _tidy(reasoning), answer, True, reason
        # Heading written, then nothing -- cut off at exactly the wrong moment.
        return _tidy(reasoning), "", False, "cut_off_at_heading"

    # 2. No answer heading anywhere. Did it start reasoning at all?
    tm = _THINK_MARKER.search(raw)
    if tm is None:
        # It ignored the format. If the text is not a list of steps, it simply
        # answered -- which is the baseline behaviour and perfectly scorable, so keep
        # ALL of it. Taking only the last paragraph here would throw away half a
        # good answer.
        if len(_STEP_LINE.findall(raw)) < 2:
            return "", _tidy(raw), True, "answered_without_headings"
        return _tidy(raw), _last_paragraph(raw), False, "steps_without_headings"

    # 3. Reasoning started and the answer section never arrived.
    body = raw[tm.end():]
    return (_tidy(body), _last_paragraph(body), False,
            "ran_out_of_room" if truncated else "no_answer_section")


# ---------- Arabic text comparison (shared with the sanity-check cell) ----------
_DIACRITICS = dict.fromkeys(range(0x064B, 0x0653))


def normalise(s):
    """Arabic forms differ cosmetically between the tafsir books and the model's
    output -- diacritics, alef and ya variants, tatweel. Compare on a common form or
    any overlap measure reports false alarms."""
    s = unicodedata.normalize("NFKC", s).translate(_DIACRITICS)
    for a, b in (("أإآٱ", "ا"), ("ى", "ي"), ("ة", "ه"), ("ـ", "")):
        for ch in a:
            s = s.replace(ch, b)
    return "".join(ch if ch.isalnum() or ch.isspace() else " " for ch in s)


def content_words(s):
    # 4+ characters skips particles and pronouns, which match everything.
    return {w for w in normalise(s).split() if len(w) >= 4}


# ---------- the worked example's reasoning ----------
# The example's four reasoning lines are filled in from real fields of the question's own
# record and from the passages the retriever really returned. Writing them by hand would
# mean inventing a Tafsir sentence, which is exactly what the model is being told never
# to do -- and the model imitates what it is shown, so a fabricated example teaches
# fabrication.
_ARABIC_DIGITS = str.maketrans("0123456789", "٠١٢٣٤٥٦٧٨٩")


def _ar(n):
    """123 -> '١٢٣'. The prompt's own numbering is Arabic-Indic; the example matches it."""
    return str(n).translate(_ARABIC_DIGITS)


def verse_range(item):
    """(start, end) verse numbers from a QA record, or (None, None). They are floats in
    the source JSON, and occasionally missing."""
    out = []
    for key in ("Verse_Number_start", "Verse_Number_End"):
        v = item.get(key)
        try:
            out.append(int(float(v)))
        except (TypeError, ValueError):
            out.append(None)
    return tuple(out)


def support_sentence(passage_text, reference, max_chars=180):
    """The one sentence of the retrieved passage that most overlaps the reference
    answer -- i.e. the sentence the answer was actually taken from."""
    ref = content_words(reference)
    sents = [s.strip() for s in re.split(r"[.\n؛؟!]", passage_text) if len(s.strip()) > 20]
    if not sents:
        return ""
    best = max(sents, key=lambda s: len(ref & content_words(s)))
    return (best[:max_chars].rstrip() + " ...") if len(best) > max_chars else best


def shot_reasoning(item, hit_numbers, support):
    """The four reasoning lines of the worked example, in the exact shape the CoT
    instructions ask for: what is being asked, which numbered passages answer it, what
    in them supports the answer, and the grounding check."""
    sura = str(item.get("SURA_name") or "").strip()
    start, end = verse_range(item)
    where = f"سورة {sura}" if sura else "السورة المذكورة"
    if start:
        where += f"، الآية {_ar(start)}" + (f"-{_ar(end)}" if end and end != start else "")
    topic = str(item.get("Q_type2_Arabic") or "").strip() or "المعنى"
    nums = "، ".join("المقطع " + _ar(n) for n in hit_numbers) or "المقاطع المعروضة"
    return "\n".join([
        f"١. السؤال عن {where}؛ والمطلوب: {topic}.",
        f"٢. الإجابة موجودة في {nums}؛ وبقية المقاطع لا تتناول ما سُئل عنه.",
        f"٣. جاء فيه: {support}" if support else "٣. نص المقطع يذكر ما تقوم عليه الإجابة.",
        "٤. كل ما ورد في الإجابة النهائية مذكور صراحة في السياق، فلا موضع لـ'لا أعرف'.",
    ])


print("Chain-of-thought prompting:", "ON" if USE_COT else "OFF (paper baseline)")
print("One-shot example:", f"ON ({N_SHOT} example)" if USE_ONE_SHOT else "OFF (zero-shot)")
print("-" * 72)
print(ACTIVE_SYSTEM_PROMPT)

In [ ]:
# ========== RETRIEVAL + GENERATION ==========
# Retrieval is untouched: same chunks, same e5-small embedder, same cosine similarity,
# same k=6 as the paper's best row. CoT and the one-shot example change the generator only.
import random

ONE_SHOT = []          # filled in at the bottom of this cell when USE_ONE_SHOT is on


def retrieve(question, k=TOP_K):
    qv = np.array(embed_model.encode(["query: " + question], normalize_embeddings=True),
                  dtype="float32")
    _, ids = index.search(qv, k)
    return [chunks[i] for i in ids[0]]


def _chat_text(user_prompt, system_prompt, shots=()):
    """Render the prompt with the chat template. Each shot is a (user, assistant) pair
    and becomes a real completed turn, which is what few-shot prompting means for an
    instruction-tuned model -- the example sits where a previous answer would have sat,
    not inside the question."""
    turns = []
    for shot_user, shot_reply in shots:
        turns += [{"role": "user", "content": shot_user},
                  {"role": "assistant", "content": shot_reply}]
    turns.append({"role": "user", "content": user_prompt})

    if SUPPORTS_SYSTEM_ROLE:
        msgs = [{"role": "system", "content": system_prompt}] + turns
    else:
        msgs = list(turns)
        msgs[0] = {**msgs[0], "content": f"{system_prompt}\n\n{msgs[0]['content']}"}

    try:
        return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except Exception as e:
        # A few chat templates reject multi-turn input. Falling back to one turn with the
        # example written into it keeps the run alive; the example is still shown, just
        # not as a separate turn.
        if not shots:
            raise
        print(f"[note] chat template rejected multi-turn input ({type(e).__name__}); "
              "writing the example into the question instead.")
        inline = "".join(f"مثال محلول:\n{u}\n{a}\n\n{'-' * 40}\n\n" for u, a in shots)
        return _chat_text(inline + "الآن أجب عن السؤال التالي:\n" + user_prompt,
                          system_prompt)


def _user_message(top, question, chars, numbered):
    """The Context/Question/Answer block. Used for BOTH the real question and the worked
    example, so the example can never drift into a shape the model is not asked for."""
    parts = []
    for i, c in enumerate(top, 1):
        t = c["chunk_text"]
        t = (t[:chars] + " ...") if len(t) > chars else t
        # Numbering the passages gives step 2 of the reasoning something to point at.
        # "i of n" is spelled out because the model otherwise cites invented numbers.
        parts.append(f"[المقطع {i} من {len(top)}]\n{t}" if numbered else t)
    return f"Context: {chr(10).join(parts)}\nQuestion: {question}\nAnswer:"


def _assemble(top, question, chars, cot, shots=()):
    return _chat_text(_user_message(top, question, chars, cot),
                      system_prompt_for(cot, bool(shots)), shots)


def _fit(top, question, cot, shots=()):
    """Shrink the CONTEXT until the prompt fits, so the question is never cut off --
    it sits after the context, so truncating the prompt would delete it.

    Returns (text, chars, shots_kept). The worked example is dropped only if even a
    150-character context does not fit alongside it; that is recorded per answer rather
    than silently swallowed, because an answer produced without the example is not
    really part of a one-shot run."""
    for c in [MAX_CHARS_PER_CHUNK, 1200, 800, 500, 300, 150]:
        text = _assemble(top, question, c, cot, shots)
        if len(tokenizer(text)["input_ids"]) <= MAX_PROMPT_TOKENS:
            return text, c, bool(shots)
    if shots:
        return _fit(top, question, cot, ())
    return text, 150, False


def _run(text, max_new_tokens):
    """Returns (generated_text, truncated).

    truncated=True means generation stopped because it hit the cap, not because the
    model finished. That distinction decides whether a retry is worth a minute of GPU.
    """
    for _ in range(3):
        try:
            inputs = tokenizer(text, return_tensors="pt").to(model.device)
            torch.cuda.empty_cache()
            with torch.no_grad():
                # do_sample=False is greedy decoding -- the paper's temperature=0.0.
                out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                     do_sample=False)
            new = out[0][inputs["input_ids"].shape[1]:]
            gen = tokenizer.decode(new, skip_special_tokens=True).strip()
            return gen, len(new) >= max_new_tokens
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            max_new_tokens = max(256, max_new_tokens // 2)
            print(f"    [out of memory -- retrying with max_new_tokens={max_new_tokens}]")
    raise RuntimeError("Out of memory even at the smallest generation budget.")


def generate(top, question, cot=None, shots=None):
    """Returns {prediction, reasoning, raw, parsed_ok, parse_reason, n_shot_used}.

    prediction is what gets scored.
    """
    cot = USE_COT if cot is None else cot
    shots = (ONE_SHOT if USE_ONE_SHOT else []) if shots is None else shots
    text, chars, kept = _fit(top, question, cot, shots)
    n_shot_used = len(shots) if kept else 0

    try:
        raw, truncated = _run(text, MAX_NEW_TOKENS)
    except RuntimeError:
        # Last resort: squeeze the context instead of the generation budget.
        text = _assemble(top, question, 150, cot, shots if kept else ())
        raw, truncated = _run(text, MAX_NEW_TOKENS)

    if not cot:
        return {"prediction": raw, "reasoning": "", "raw": raw, "parsed_ok": True,
                "parse_reason": "baseline", "n_shot_used": n_shot_used}

    reasoning, answer, parsed_ok, reason = split_cot(raw, truncated)

    # Retry ONLY when the model actually ran out of room. Decoding is greedy, so a
    # model that stopped on its own regenerates the identical text -- the retry would
    # burn a minute per question to reach the same conclusion, which over a whole run
    # is hours of GPU for nothing.
    if not parsed_ok and truncated:
        raw2, trunc2 = _run(text, COT_RETRY_NEW_TOKENS)
        r2, a2, ok2, reason2 = split_cot(raw2, trunc2)
        if ok2 or (a2 and not answer):
            reasoning, answer, raw, parsed_ok, reason = r2, a2, raw2, ok2, reason2

    return {"prediction": answer, "reasoning": reasoning, "raw": raw,
            "parsed_ok": parsed_ok, "parse_reason": reason, "n_shot_used": n_shot_used}


def ask(question, verbose=True):
    top = retrieve(question)
    if verbose:
        for i, c in enumerate(top, 1):
            print(f"[{i}] {c['source']} · {c['surah_name']} {c['start_ayah']}-{c['end_ayah']}")
    res = generate(top, question)
    res["sources"] = [f"{c['source']}:{c['surah_number']}:"
                      f"{c['start_ayah']}-{c['end_ayah']}" for c in top]
    if verbose:
        if res["reasoning"]:
            print("\n--- التفكير (not scored) ---\n" + res["reasoning"])
        print("\n--- الإجابة النهائية (scored) ---\n" + res["prediction"])
        print(f"\n[parse: {res['parse_reason']}]")
        if not res["parsed_ok"]:
            print("[warn] no usable answer heading -- fell back to the last paragraph.")
    return res


# ================= THE ONE-SHOT EXAMPLE =================


def holdout_shot_pool(items, size=SHOT_POOL_SIZE, seed=SEED):
    """Split the Direct questions into (example pool, evaluation pool).

    The reserved questions are removed from the evaluation pool in EVERY mode, one-shot
    or not. Two reasons, and both matter for the numbers being believable:
      * the question used as the example is never also a question being scored, so the
        model is never shown the reference answer of something it is then graded on;
      * a zero-shot run and a one-shot run at the same seed answer the identical
        questions, so the difference between them is the prompt and nothing else.
    Its own RNG, so changing N_EVAL_SAMPLES does not change which questions are reserved.
    """
    reserved = set(random.Random(1000 + seed).sample(range(len(items)),
                                                     min(size, len(items))))
    return ([items[i] for i in sorted(reserved)],
            [d for i, d in enumerate(items) if i not in reserved])


def _shot_passages(top, reference, m=SHOT_MAX_PASSAGES):
    """Cut the k retrieved passages down to the m shown in the example, keeping the one
    the answer actually came from. Returns (passages, its 1-based number, coverage).

    coverage is the share of the reference answer's content words that appear in that
    passage -- it is how a good example is chosen rather than assumed."""
    ref = content_words(reference)
    overlap = [len(ref & content_words(c["chunk_text"])) for c in top]
    best = max(range(len(top)), key=lambda i: overlap[i])
    keep = (list(range(min(m, len(top)))) if best < m
            else list(range(m - 1)) + [best])
    return [top[i] for i in keep], keep.index(best) + 1, overlap[best] / max(1, len(ref))


def choose_shot_items(pool, n_shot):
    """Pick the example question(s): the ones whose reference answer is short enough to
    be worth imitating AND is genuinely contained in what the retriever returns. An
    example whose 'answer' is not in its own context would teach the model to answer
    from memory -- the single behaviour this prompt exists to prevent."""
    scored = []
    for item in pool:
        ref = str(item.get("Answer") or "").strip()
        n_words = len(ref.split())
        if not SHOT_MIN_REF_WORDS <= n_words <= SHOT_MAX_REF_WORDS:
            continue
        top = retrieve(item["Question"])
        passages, hit_no, coverage = _shot_passages(top, ref)
        scored.append((coverage, item, passages, hit_no))
    if not scored:
        raise RuntimeError("No usable example in the held-out pool -- raise "
                           "SHOT_MAX_REF_WORDS or SHOT_POOL_SIZE.")
    # Best-grounded first; shorter answer breaks ties; the question text breaks the rest,
    # so the choice is the same on every machine and every rerun.
    scored.sort(key=lambda t: (-t[0], len(t[1]["Answer"].split()), t[1]["Question"]))
    picked = scored[:n_shot]
    if picked[0][0] < 0.5:
        print(f"[warn] the best example only covers {picked[0][0]:.0%} of its reference "
              "answer. It still shows the format, but it is a weak demonstration of "
              "grounding.")
    return picked


def build_shots(picked, cot):
    """Turn the chosen questions into (user turn, assistant turn) pairs."""
    shots = []
    for coverage, item, passages, hit_no in picked:
        ref = str(item["Answer"]).strip()
        user = _user_message(passages, item["Question"], SHOT_CHARS_PER_CHUNK, cot)
        if cot:
            support = support_sentence(passages[hit_no - 1]["chunk_text"], ref)
            reply = ("التفكير:\n" + shot_reasoning(item, [hit_no], support)
                     + "\n\nالإجابة النهائية:\n" + ref)
        else:
            # The baseline prompt asks for an answer and nothing else, so the example
            # shows exactly that.
            reply = ref
        shots.append((user, reply))
    return shots


direct_questions = [d for d in qa_data if d.get("Task") == "Direct" and d.get("Answer")]
shot_pool, eval_pool = holdout_shot_pool(direct_questions)
SHOT_QUESTIONS = []

print(f"Direct questions available: {len(direct_questions)} "
      f"-> {len(shot_pool)} reserved for the example, {len(eval_pool)} available to score.\n")

if USE_ONE_SHOT:
    picked = choose_shot_items(shot_pool, N_SHOT)
    ONE_SHOT = build_shots(picked, USE_COT)
    SHOT_QUESTIONS = [item["Question"] for _, item, _, _ in picked]
    print("=" * 72)
    print(f"THE WORKED EXAMPLE ({N_SHOT} shown before every question)")
    print("=" * 72)
    for (coverage, item, passages, hit_no), (user, reply) in zip(picked, ONE_SHOT):
        print(f"question: {item['Question']}")
        print(f"reference answer is {coverage:.0%} contained in passage {hit_no} of "
              f"the {len(passages)} shown")
        print(f"\n--- user turn ({len(tokenizer(user)['input_ids'])} tokens) ---")
        print(user[:900] + (" ..." if len(user) > 900 else ""))
        print("\n--- assistant turn (what the model is shown as a good reply) ---")
        print(reply)
    print("=" * 72)
else:
    print("Zero-shot: instructions only, no example.\n")

print("\n--- one test question ---")
print("Q:", eval_pool[0]["Question"], "\n")
_ = ask(eval_pool[0]["Question"])

In [ ]:
# ========== ANSWER THE QUESTIONS ==========
# Saved after every question. If this stops early, just run it again --
# it continues from where it left off instead of starting over.
# The filename carries the mode, so a CoT run, a one-shot run and a baseline run never
# overwrite each other, and the same seed makes them answer the identical questions.
import glob
import random
from collections import Counter
from tqdm.auto import tqdm

model_tag = MODEL_ID.split("/")[-1]
STEM = f"{model_tag}_{CORPUS}_{MODE}_n{N_EVAL_SAMPLES}_seed{SEED}"
path = os.path.join(RESULTS_DIR, f"preds_{STEM}{SHARD_TAG}.json")

# Drawn from eval_pool, never from the questions reserved for the worked example, and
# drawn the same way in every mode -- so the baseline, the CoT run and the one-shot run
# are scored on exactly the same 50 questions.
random.seed(SEED)
full = (list(eval_pool) if N_EVAL_SAMPLES is None else
        random.sample(eval_pool, min(N_EVAL_SAMPLES, len(eval_pool))))

if N_SHARDS > 1:
    lo, hi = shard_bounds(len(full), SHARD, N_SHARDS)
    sample = full[lo:hi]
    print(f"Shard {SHARD}/{N_SHARDS}: questions {lo + 1}-{hi} of {len(full)} "
          f"({len(sample)} to answer).")
else:
    sample = full

leaked = [q for q in SHOT_QUESTIONS if q in {i["Question"] for i in sample}]
if leaked:
    raise RuntimeError(f"{len(leaked)} example question(s) are also in the evaluation "
                       "set -- the holdout is broken, do not score this run.")


def save_answers(recs):
    """Save so that losing power mid-write cannot destroy the run.

    A plain open(path, "w") truncates the file first, so a cut during the write leaves
    a half-written file that json.load rejects -- and every answer so far is gone, not
    just the last one. Instead: write a temp file and fsync it, rotate the current file
    to .bak, then rename the temp into place. Renames are atomic, so at any instant at
    least one of path / path.bak is a complete file.
    """
    tmp, bak = path + ".tmp", path + ".bak"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(recs, f, ensure_ascii=False, indent=1)
        f.flush()
        os.fsync(f.fileno())          # force it to disk, not just the OS cache
    if os.path.exists(path):
        os.replace(path, bak)
    os.replace(tmp, path)


def load_answers():
    """Collect saved answers from everywhere they might be, newest source first:
    this session's file, its backup, then any attached Kaggle dataset.

    Combining them rather than picking one means a run can be carried across sessions
    without the Persistence setting -- download the file, upload it as a Dataset,
    attach it, and this picks up where the last session stopped. A question already
    answered by an earlier source is never redone.
    """
    base = os.path.basename(path)
    sources = [p for p in (path, path + ".bak") if os.path.exists(p)]
    sources += [p for p in sorted(glob.glob(f"/kaggle/input/**/{base}", recursive=True))
                if p not in sources]

    merged = {}
    for cand in sources:
        try:
            recs = json.load(open(cand, encoding="utf-8"))
        except (json.JSONDecodeError, UnicodeDecodeError):
            print(f"[warn] {cand} is unreadable (a save was probably interrupted) "
                  "-- trying the next source.")
            continue
        added = 0
        for r in recs:
            if r["question"] not in merged:
                merged[r["question"]] = r
                added += 1
        if cand != path and added:
            label = "backup" if cand.endswith(".bak") else "attached dataset"
            print(f"[recovered] {added} answers from the {label}: {cand}")
    return list(merged.values())


done = {r["question"]: r for r in load_answers()}
if done:
    pct = 100 * len(done) / max(1, len(sample))
    print(f"Resuming: {len(done)}/{len(sample)} already answered ({pct:.0f}% done), "
          f"{len(sample) - len(done)} to go.")

records = []
for item in tqdm(sample, desc=f"answering ({MODE})"):
    q = item["Question"]
    if q in done:
        records.append(done[q])
        continue
    res = ask(q, verbose=False)
    records.append({"question": q, "reference": item["Answer"],
                    "prediction": res["prediction"], "reasoning": res["reasoning"],
                    "raw": res["raw"], "parsed_ok": res["parsed_ok"],
                    "parse_reason": res["parse_reason"],
                    "sources": res["sources"], "mode": MODE,
                    # How many examples this particular prompt actually carried. It is
                    # N_SHOT unless the context was so long the example had to be
                    # dropped to fit, and that has to be visible, not assumed.
                    "n_shot_used": res["n_shot_used"],
                    # Stamped on every record so the merge cell can prove the two
                    # halves came from the same draw before combining them.
                    "seed": SEED, "n_total": N_EVAL_SAMPLES,
                    "shard": SHARD, "n_shards": N_SHARDS, "corpus": CORPUS})
    save_answers(records)

n_bad = sum(1 for r in records if not r.get("parsed_ok", True))
n_empty = sum(1 for r in records if not r["prediction"].strip())
print(f"\n{len(records)} answers saved to {path}")

if USE_ONE_SHOT:
    dropped = sum(1 for r in records if not r.get("n_shot_used"))
    print(f"\nWorked example: carried by {len(records) - dropped}/{len(records)} prompts"
          + (f", dropped from {dropped} that would not fit in "
             f"{MAX_PROMPT_TOKENS} tokens." if dropped else "."))
    if dropped:
        print("Those answers are effectively zero-shot. Lower SHOT_MAX_PASSAGES or "
              "SHOT_CHARS_PER_CHUNK, or raise MAX_PROMPT_TOKENS, and rerun them.")

if USE_COT:
    # Which route each answer took. Counting failures alone tells you nothing about
    # what to change; the reason tells you whether to give it more tokens or to
    # tighten the prompt.
    reasons = Counter(r.get("parse_reason", "unknown") for r in records)
    WHAT = {
        "ok": "clean 'الإجابة النهائية:' heading",
        "short_heading": "shortened heading, e.g. 'الإجابة:' -- parsed fine",
        "answered_without_headings": "ignored the format and just answered -- kept in full",
        "steps_without_headings": "wrote steps with no headings -- last paragraph used",
        "cut_off_at_heading": "stopped right after the heading -- no answer text",
        "ran_out_of_room": "hit the token cap mid-reasoning -- raise MAX_NEW_TOKENS",
        "no_answer_section": "finished without an answer section -- prompt problem",
    }
    print("\nHow each answer was parsed:")
    for k, v in reasons.most_common():
        flag = " " if k in ("ok", "short_heading", "answered_without_headings") else "!"
        print(f" {flag} {v:5}  {k:26} {WHAT.get(k, '')}")
    print(f"\nUsable: {len(records) - n_bad}/{len(records)}.")
    if reasons["ran_out_of_room"] or reasons["cut_off_at_heading"]:
        print("Raise MAX_NEW_TOKENS -- some answers are being cut off.")

if n_empty:
    print(f"[warn] {n_empty} empty predictions -- they will score 0.")

In [ ]:
# ========== THE RESULT ==========
# evaluate() is a function so the merge cell below can score a combined set with
# exactly the same code -- two copies of a metric block is how halves end up scored
# differently from the whole.
import sacrebleu
from bert_score import score as bert_score

PROMPT_LABEL = ("chain-of-thought" if USE_COT else "paper baseline prompt") + (
    f" + {N_SHOT}-shot example" if USE_ONE_SHOT else " (zero-shot)")


def evaluate(recs, label, save_as=None):
    preds = [r["prediction"] for r in recs]
    refs = [r["reference"] for r in recs]

    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score
    P, R, F1 = bert_score(preds, refs, model_type=BERTSCORE_MODEL, verbose=False)
    p, r, f1 = P.mean().item()*100, R.mean().item()*100, F1.mean().item()*100

    avg_pred_len = sum(len(x.split()) for x in preds) / max(1, len(preds))
    avg_ref_len = sum(len(x.split()) for x in refs) / max(1, len(refs))
    idk = sum(1 for x in preds
              if "لا أعرف" in x or "لا اعرف" in x or "I don't know" in x)
    bad = sum(1 for x in recs if not x.get("parsed_ok", True))
    no_shot = sum(1 for x in recs if not x.get("n_shot_used"))

    print("=" * 72)
    print(f"  OntologyRAG-Q best configuration — {MODEL_ID}")
    print(f"  {label}")
    print("=" * 72)
    print(f"  BLEU             {bleu:6.2f}")
    print(f"  CHRF             {chrf:6.2f}")
    print(f"  BERT Precision   {p:6.2f}")
    print(f"  BERT Recall      {r:6.2f}")
    print(f"  BERT F1          {f1:6.2f}")
    print("=" * 72)
    print(f"  {len(recs)} questions · {QUANT_BITS}-bit · {len(chunks)} chunks "
          f"· corpus={CORPUS} · seed {SEED}")
    print(f"  mode={MODE} · avg answer {avg_pred_len:.0f} words vs reference "
          f"{avg_ref_len:.0f} · 'لا أعرف' on {idk}/{len(preds)}")
    if USE_ONE_SHOT:
        print(f"  example carried by {len(recs) - no_shot}/{len(recs)} prompts")
    print(f"  BERTScore model: {BERTSCORE_MODEL}")
    print()

    print("Table 4 row:")
    print(f"{'LLM':<16}{'Chunk':<16}{'Parameters':<48}{'Embed':<10}"
          f"{'BLEU':>7}{'CHRF':>7}{'Prec':>7}{'Rec':>7}{'F1':>7}")
    print("-" * 125)
    params = ("k:6, Similarity, temperature=0.0"
              + (", CoT" if USE_COT else "")
              + (f", {N_SHOT}-shot" if USE_ONE_SHOT else ""))
    print(f"{model_tag:<16}{'Ayat ontology':<16}{params:<48}{'E5-small':<10}"
          f"{bleu:>7.2f}{chrf:>7.2f}{p:>7.2f}{r:>7.2f}{f1:>7.2f}")

    out = {"model": MODEL_ID, "quant_bits": QUANT_BITS, "corpus": CORPUS,
           "mode": MODE, "use_cot": USE_COT, "use_one_shot": USE_ONE_SHOT,
           "n_shot": N_SHOT if USE_ONE_SHOT else 0,
           "n_prompts_without_example": no_shot,
           "shot_questions": SHOT_QUESTIONS,
           "concise_final_answer": CONCISE_FINAL_ANSWER,
           "label": label, "n": len(recs), "seed": SEED, "n_chunks": len(chunks),
           "top_k": TOP_K, "bertscore_model": BERTSCORE_MODEL,
           "avg_pred_words": avg_pred_len, "avg_ref_words": avg_ref_len,
           "n_idk": idk, "n_parse_fallback": bad,
           "parse_reasons": (dict(Counter(x.get("parse_reason", "unknown")
                                          for x in recs)) if USE_COT else {}),
           "BLEU": bleu, "CHRF": chrf, "BERT_P": p, "BERT_R": r, "BERT_F1": f1}
    if save_as:
        json.dump(out, open(save_as, "w"), indent=2)
        print(f"\nSaved to {save_as}")
    return out


mode_label = PROMPT_LABEL
if N_SHARDS > 1:
    mode_label += f" — YOUR HALF ONLY (shard {SHARD} of {N_SHARDS})"

RESULT_PATH = os.path.join(
    RESULTS_DIR, f"result_{model_tag}_{CORPUS}_{MODE}_n{len(records)}{SHARD_TAG}.json")
result = evaluate(records, mode_label, RESULT_PATH)

if N_SHARDS > 1:
    print(f"\nThis is {len(records)} questions, not the full {N_EVAL_SAMPLES}. "
          "Do not report it as the")
    print("final result -- the merge cell below produces that once both halves exist.")

In [ ]:
# ========== SANITY CHECKS: WHOLE CORPUS, GROUNDED ANSWERS, CLEAN EXAMPLE ==========
# Three questions this cell answers with evidence rather than assurance:
#   1. Did the search really cover all 15 books and all 55,471 chunks?
#   2. Is each answer actually built from the retrieved passages, or invented?
#   3. Is the one-shot example separate from what is being scored, and did it reach
#      every prompt?
# No GPU needed -- it re-reads what the run already saved.
problems, notes = [], []

# ---- 1. corpus coverage ----
books_in_index = sorted({c["source"] for c in chunks})
if CORPUS == "all":
    if len(books_in_index) != 15:
        problems.append(f"{len(books_in_index)} books in the index, expected 15: "
                        f"{books_in_index}")
    if len(chunks) != 55471:
        problems.append(f"{len(chunks)} chunks, expected 55,471 (paper Table 6)")
if index.ntotal != len(chunks):
    problems.append(f"index holds {index.ntotal} vectors for {len(chunks)} chunks -- "
                    "retrieval would return the wrong passages")

per_book = Counter(c["source"] for c in chunks)
print(f"Corpus: {len(books_in_index)} books, {len(chunks)} chunks, "
      f"{index.ntotal} vectors searched per question.")
for b, n in sorted(per_book.items()):
    print(f"  {b:20} {n:6}")
empty_books = [b for b in books_in_index if per_book[b] == 0]
if empty_books:
    problems.append(f"books contributing nothing: {empty_books}")


# ---- 2. are the answers grounded in the retrieved text? ----
# normalise() and content_words() come from the prompt cell, so the example was chosen
# with the same word-overlap measure the answers are audited with.
#
# The records store retrieved passages as "book:surah:start-end"; rebuild the lookup so
# each answer can be compared against the exact text its own question retrieved.
by_id = {}
for c in chunks:
    by_id.setdefault(
        f"{c['source']}:{c['surah_number']}:{c['start_ayah']}-{c['end_ayah']}",
        c["chunk_text"])

scores, unmatched_ids = [], 0
for rec in records:
    ctx = []
    for sid in rec.get("sources", []):
        if sid in by_id:
            ctx.append(by_id[sid])
        else:
            unmatched_ids += 1
    words = content_words(rec["prediction"])
    if not words or not ctx:
        continue
    ctx_words = content_words(" ".join(ctx))
    scores.append((len(words & ctx_words) / len(words), rec))

if scores:
    mean = sum(s for s, _ in scores) / len(scores)
    weak = sorted(scores, key=lambda x: x[0])[:5]
    low = sum(1 for s, _ in scores if s < 0.5)
    print(f"\nGrounding: on average {mean * 100:.0f}% of the words in an answer also "
          "appear in the")
    print(f"passages that were retrieved for that question ({len(scores)} answers "
          "measured).")
    print(f"Answers under 50%: {low} ({100 * low / len(scores):.0f}%). These are the "
          "ones to eyeball --")
    print("a low score means the wording came from the model, not from the tafsir.")
    print("\nWeakest five:")
    for s, rec in weak:
        print(f"  {s * 100:3.0f}%  {rec['question'].strip()[:66]}")
    notes.append(f"mean grounding {mean * 100:.0f}%")
if unmatched_ids:
    problems.append(f"{unmatched_ids} retrieved passage ids not found in the current "
                    "chunks -- predictions and index disagree")


# ---- 3. the one-shot example ----
# The example carries a reference answer inside the prompt. If its question were also
# being scored, the model would be shown the answer it is graded on and the score would
# mean nothing -- so this is checked, not trusted.
if USE_ONE_SHOT:
    answered = {r["question"] for r in records}
    overlap = [q for q in SHOT_QUESTIONS if q in answered]
    if overlap:
        problems.append(f"{len(overlap)} example question(s) were also scored -- "
                        "the run leaks its own answer")
    reserved = {d["Question"] for d in shot_pool}
    if reserved & answered:
        problems.append(f"{len(reserved & answered)} held-out questions were scored")
    no_shot = [r for r in records if not r.get("n_shot_used")]
    if no_shot:
        notes.append(f"{len(no_shot)} of {len(records)} prompts were too long for the "
                     "example and ran zero-shot")
    print(f"\nOne-shot example: {len(SHOT_QUESTIONS)} question(s), none of them scored; "
          f"{len(records) - len(no_shot)}/{len(records)} prompts carried it.")
    for q in SHOT_QUESTIONS:
        print(f"  example question (held out): {q.strip()[:66]}")


# ---- 4. integrity of the answer set ----
qs = [r["question"] for r in records]
if len(set(qs)) != len(qs):
    problems.append(f"{len(qs) - len(set(qs))} duplicated questions in the results")
missing_ref = sum(1 for r in records if not r.get("reference", "").strip())
if missing_ref:
    problems.append(f"{missing_ref} records have no reference answer to score against")
empty_pred = sum(1 for r in records if not r["prediction"].strip())
if empty_pred:
    problems.append(f"{empty_pred} empty predictions (they score 0)")
if len(records) != (N_EVAL_SAMPLES if N_SHARDS == 1 else len(sample)):
    notes.append(f"answered {len(records)} of {N_EVAL_SAMPLES} -- run is incomplete")
modes = {r.get("mode") for r in records}
if modes != {MODE}:
    problems.append(f"records mix prompt modes {sorted(modes)} -- a stale predictions "
                    f"file is being reused; expected only {MODE!r}")

print("\n" + "=" * 72)
if problems:
    print("PROBLEMS FOUND -- do not report these numbers until they are resolved:")
    for p in problems:
        print("  !", p)
else:
    print("All checks passed: full corpus searched, no duplicates, every answer scored")
    print("against a reference, every retrieved passage traceable to the index, and the")
    print("worked example held out of the scored set.")
for n in notes:
    print("  -", n)
print("=" * 72)

## Merging the halves (only if you split the run)

`N_SHARDS = 1` here, so this section does nothing — skip it. It matters only if two people
split one evaluation set between them.

Put your teammate's predictions file into the `results/` folder next to yours — upload it
as a Kaggle dataset, or download both and run this cell locally. Then run the cell below.

It refuses to merge halves that were not drawn from the same set: it checks the seed, the
total size, the corpus, the model and the mode (which includes the prompt setting, so a
CoT half can never be merged with a one-shot half) on every record, and it checks that no
question appears in both halves. **The combined number is the one you report.** Scoring
half the set and doubling nothing is not the same as scoring the whole — BLEU and CHRF are
corpus-level metrics, so they have to be computed over the whole set at once.

In [ ]:
# ========== MERGE THE SHARDS AND SCORE THE FULL SET ==========
import glob

if N_SHARDS <= 1:
    print("Not a split run (N_SHARDS = 1) -- nothing to merge.")
else:
    found, missing = {}, []
    for s in range(1, N_SHARDS + 1):
        f = os.path.join(RESULTS_DIR, f"preds_{STEM}_shard{s}of{N_SHARDS}.json")
        (found.setdefault(s, f) if os.path.exists(f) else missing.append((s, f)))

    for s, f in sorted(found.items()):
        n = len(json.load(open(f, encoding="utf-8")))
        print(f"[found]   shard {s}: {n:5} answers  {f}")
    for s, f in missing:
        print(f"[missing] shard {s}: {f}")

    if missing:
        print(f"\n{len(missing)} of {N_SHARDS} halves missing. Copy your teammate's "
              "predictions file into")
        print(f"{RESULTS_DIR}/ under exactly that name, then run this cell again.")
    else:
        merged, seen, problems = [], {}, []
        for s, f in sorted(found.items()):
            for rec in json.load(open(f, encoding="utf-8")):
                # Every record states the draw it came from. Any disagreement means the
                # two halves are not halves of one 2000, and merging would produce a
                # number that looks fine and means nothing.
                for field, mine in (("seed", SEED), ("n_total", N_EVAL_SAMPLES),
                                    ("n_shards", N_SHARDS), ("corpus", CORPUS),
                                    ("mode", MODE)):
                    theirs = rec.get(field, "<missing>")
                    if theirs != mine:
                        problems.append(f"shard {s}: {field} is {theirs!r}, "
                                        f"yours is {mine!r}")
                q = rec["question"]
                if q in seen:
                    problems.append(f"question answered in both shard {seen[q]} "
                                    f"and shard {s}")
                else:
                    seen[q] = s
                    merged.append(rec)

        if problems:
            print("\nCannot merge -- the halves do not match:")
            for p in sorted(set(problems))[:10]:
                print("  -", p)
            print("\nBoth notebooks need identical SEED, N_EVAL_SAMPLES, CORPUS, "
                  "USE_COT and MODEL_ID.")
        else:
            print(f"\nMerged {len(merged)} unique answers from {N_SHARDS} halves.")
            if len(merged) != N_EVAL_SAMPLES:
                print(f"[warn] expected {N_EVAL_SAMPLES}. A half is incomplete -- "
                      "whoever owns it")
                print("       should run their notebook again to finish it, then "
                      "re-merge.")
            print()
            merged_path = os.path.join(
                RESULTS_DIR, f"result_{model_tag}_{CORPUS}_{MODE}_n{len(merged)}_MERGED.json")
            merged_result = evaluate(
                merged, f"{PROMPT_LABEL} — COMBINED {len(merged)} questions "
                        f"({N_SHARDS} team members)", merged_path)
            print("\nThis is the row to report.")

In [ ]:
# ========== ALL PROMPT SETTINGS, SIDE BY SIDE ==========
# Fills itself in from whatever has been run at this n. Flip USE_COT / USE_ONE_SHOT and
# Run All to add a row; results are written to one file per mode, so nothing is
# overwritten. The four rows of the ablation are:
#     base         paper prompt, no example        (zero-shot baseline)
#     base_1shot   paper prompt + one example
#     cot          reason then answer, no example
#     cot_1shot    reason then answer + one example   <- this notebook
ORDER = ["base", "base_1shot", "cot", "cot_1shot"]
NICE = {"base": "paper baseline", "base_1shot": "baseline + 1-shot",
        "cot": "chain-of-thought", "cot_1shot": "CoT + 1-shot"}

rows = []
for m in ORDER + [MODE]:
    if any(m == r[0] for r in rows):
        continue
    f = os.path.join(RESULTS_DIR,
                     f"result_{model_tag}_{CORPUS}_{m}_n{len(records)}.json")
    if m == MODE:
        rows.append((m, result))
    elif os.path.exists(f):
        rows.append((m, json.load(open(f))))

keys = ["BLEU", "CHRF", "BERT_P", "BERT_R", "BERT_F1"]
print(f"{'prompt':<22}" + "".join(f"{k:>10}" for k in keys) + f"{'words':>8}")
print("-" * 80)
for m, d in rows:
    mark = "  <- this run" if m == MODE else ""
    print(f"{NICE.get(m, m):<22}" + "".join(f"{d[k]:>10.2f}" for k in keys)
          + f"{d['avg_pred_words']:>8.0f}" + mark)

base = dict(rows).get("base")
if base and len(rows) > 1:
    print("-" * 80)
    for m, d in rows:
        if m == "base":
            continue
        print(f"{'vs baseline: ' + NICE.get(m, m):<22}"
              + "".join(f"{d[k] - base[k]:>+10.2f}" for k in keys)
              + f"{d['avg_pred_words'] - base['avg_pred_words']:>+8.0f}")

missing = [m for m in ORDER if m not in dict(rows)]
if missing:
    print(f"\nNot run yet at n={len(records)}: {', '.join(missing)}.")
    print("Set USE_COT / USE_ONE_SHOT accordingly and Run All to fill in a row.")
print(f"\nEvery row: the same {len(records)} questions (seed {SEED}), corpus={CORPUS}, "
      f"k={TOP_K}, greedy decoding,")
print(f"and the same {SHOT_POOL_SIZE} questions held out of all of them.")

In [ ]:
# ========== LOOK AT THE REASONING ==========
# The reasoning is never scored, but it is the only way to see *why* an answer went
# wrong -- retrieval missed the verse, or the right passage was retrieved and the model
# reasoned past it. With the one-shot example on, it also shows whether the model
# followed the example's format or drifted back to its own.
if USE_COT and SHOW_TRACES:
    for rec in records[:SHOW_TRACES]:
        print("=" * 78)
        print("Q:", rec["question"])
        print("\nretrieved:", ", ".join(rec.get("sources", [])))
        print(f"example in prompt: {'yes' if rec.get('n_shot_used') else 'no'}")
        print("\n--- التفكير ---\n" + (rec.get("reasoning") or "(none)"))
        print("\n--- الإجابة النهائية (scored) ---\n" + rec["prediction"])
        print("\n--- reference ---\n" + rec["reference"])
        print()
else:
    print("Nothing to show (baseline mode, or SHOW_TRACES = 0).")